# 03 · Neural networks, mean–variance outputs, and deep ensembles

*Level: intermediate. Guide Section 7. Code: `models.MLPEnsemble`.*

**Idea.** A network stacks layers $\mathbf h_\text{out} = \mathrm{act}(W\mathbf h_\text{in} + \mathbf b)$, where $W$ and $\mathbf b$ are learned
and `act` is a fixed bend. It is trained by gradient descent (notebook 00). Our networks also predict their own error bar, and an
ensemble of five tells where they disagree.

In [ ]:
from tutorial_utils import *

import torch
_ = torch.manual_seed(0)

## 1. Activation functions

Without a nonlinear activation, stacked layers collapse into one linear map. We use **SiLU**, $u/(1+e^{-u})$, a smooth version of ReLU, $\max(0, u)$.

In [ ]:
u = np.linspace(-5, 5, 200)
plt.figure(figsize=(5, 3)); plt.plot(u, u / (1 + np.exp(-u)), label="SiLU"); plt.plot(u, np.maximum(0, u), "--", label="ReLU")
plt.legend(); plt.xlabel("u"); plt.show()

## 2. A small network, trained step by step

The data have noise that **grows** with $x$. We first train a plain network (one output, squared error) with a hand-written loop:
forward pass → loss → `backward()` (backpropagation computes every gradient) → an Adam step.

In [ ]:
rng = np.random.default_rng(0)
x = rng.uniform(0, 3, 400); noise_sd = 0.05 + 0.15 * x
y = np.sin(2 * x) + noise_sd * rng.normal(size=x.size)
xt, yt = torch.tensor(x[:, None], dtype=torch.float32), torch.tensor(y[:, None], dtype=torch.float32)

net = torch.nn.Sequential(torch.nn.Linear(1, 64), torch.nn.SiLU(), torch.nn.Linear(64, 64), torch.nn.SiLU(), torch.nn.Linear(64, 1))
opt = torch.optim.Adam(net.parameters(), lr=3e-3)
for step in range(1500):
    loss = ((net(xt) - yt) ** 2).mean()   # forward pass and loss
    opt.zero_grad(); loss.backward()       # backpropagation: d loss / d every weight
    opt.step()                             # one Adam step
    if step % 500 == 0:
        print(f"step {step:4d}  loss {loss.item():.4f}")
print("number of weights:", sum(p.numel() for p in net.parameters()))

## 3. Predicting an error bar: the Gaussian negative log-likelihood

Two outputs, $\mu(x)$ and $\log\sigma^2(x)$, trained with

$$\text{loss} = \tfrac12\left[\log\sigma^2(x_i) + \frac{(y_i - \mu(x_i))^2}{\sigma^2(x_i)}\right].$$

The first term stops it claiming huge error bars everywhere; the second stops it claiming small ones where it is wrong.

In [ ]:
class MeanVar(torch.nn.Module):
    def __init__(s):
        super().__init__()
        s.body = torch.nn.Sequential(torch.nn.Linear(1, 64), torch.nn.SiLU(), torch.nn.Linear(64, 64), torch.nn.SiLU())
        s.mu, s.logvar = torch.nn.Linear(64, 1), torch.nn.Linear(64, 1)
    def forward(s, x):
        h = s.body(x); return s.mu(h), s.logvar(h)

def train_meanvar(seed):
    torch.manual_seed(seed); m = MeanVar(); o = torch.optim.Adam(m.parameters(), lr=3e-3)
    for step in range(2000):
        mu, lv = m(xt); loss = 0.5 * (lv + (yt - mu) ** 2 / lv.exp()).mean()
        o.zero_grad(); loss.backward(); o.step()
    return m

mv = train_meanvar(0)
xs = torch.linspace(0, 3, 200)[:, None]
with torch.no_grad():
    mu, lv = mv(xs)
sd = lv.exp().sqrt().numpy()[:, 0]
plt.figure(figsize=(7, 3)); plt.scatter(x, y, s=4, color="0.6")
plt.plot(xs[:, 0], mu[:, 0], "C0"); plt.fill_between(xs[:, 0], mu[:, 0] - 2 * sd, mu[:, 0] + 2 * sd, alpha=0.25)
plt.title("learned ±2σ(x): the network found that the noise grows"); plt.show()
print("learned σ at x = 0.5, 2.5:", sd[33].round(3), sd[166].round(3), "| truth:", 0.05 + 0.15 * 0.5, 0.05 + 0.15 * 2.5)

## 4. A deep ensemble: knowing what you don't know

One network extrapolates confidently. **Five** networks from different random starts agree where data pin the function down and disagree
elsewhere. The ensemble's variance is

$$s^2 = \underbrace{\overline{\sigma_k^2}}_\text{aleatoric} + \underbrace{\mathrm{Var}_k(\mu_k)}_\text{epistemic}.$$

In [ ]:
nets = [train_meanvar(k) for k in range(5)]
xs = torch.linspace(-1.5, 4.5, 300)[:, None]
with torch.no_grad():
    outs = [n(xs) for n in nets]
mus = np.array([o[0][:, 0].numpy() for o in outs]); vars_ = np.array([o[1][:, 0].exp().numpy() for o in outs])
mean, epi, ale = mus.mean(0), mus.var(0), vars_.mean(0)
plt.figure(figsize=(8, 3)); plt.scatter(x, y, s=4, color="0.6")
for mk in mus: plt.plot(xs[:, 0], mk, lw=0.8)
plt.fill_between(xs[:, 0], mean - 2 * np.sqrt(ale + epi), mean + 2 * np.sqrt(ale + epi), alpha=0.15)
plt.axvspan(-1.5, 0, color="k", alpha=0.05); plt.axvspan(3, 4.5, color="k", alpha=0.05)
plt.ylim(-4, 4); plt.title("5 networks: they agree on 0..3 and split outside (grey)"); plt.show()

## 5. On the toy CCD

Our `models.MLPEnsemble` wraps this:
* 5 members, 2 hidden layers of 64 units;
* AdamW (Adam plus weight decay);
* early stopping on 15 % of the training rows.

In [ ]:
df, curves, Y, groups = toy_scan(120, seed=1)
X, cols = data.make_inputs(df, ("theta", "trace", "ref"), drop_constant=False)
mean, std = grouped_cv(lambda: models.MLPEnsemble(), X, Y, groups)
_ = report("NN ensemble (settings + own traces)", Y, mean, std)

## When to use it

* **Good:** smooth nonlinear relations with many inputs, and a few hundred or more examples.
* **Bad:** tens of examples; error bars far from the data are less trustworthy than a GP's.

Next: **04 · Convolutional networks on the trace curves.**